# Training an AI agent to play Centipede with reinforcement learning and a convolution neural network
Andrew Metzroth

Final project, CSPB 3202 at CU Boulder


Github link: https://github.com/bulgakovian/CS-3202-Final-q-learning-agent/tree/main

## Overview

For my project, I chose to use the Arcade Learning Environment in Gymnnasium to emulate the classic arcade game Centipede in a reinforcement learning environment. I knew that I wanted to test several different types of agents and explore their limitations. I also knew that I wanted to at least get some level of reconizable result from a learning agent. Finally, I knew I wanted to attempt something that involved deep learning and neural networks. This was also an attempt for me to expand my limited experience with coding and the use of external libraries. The work proved to be challenging, but I also was able to attempt a lot of experiments in the project. 

As a relative beginner to these areas of research, I wanted to explore multiple different approaches, and set a personal goal of "getting things working" with a variety of models with manipulable elements, rather than refining a perfect learning agent on my first try.

This was a project full of new experiences for me! Some of the the things I learned or experienced during this implementation include:
- Setting up and installing multiple external libraries on a local instance of Python on my machine. This proved to be a larger timesink than I anticipated, including a complete wipe and reinstall of python and VSCode when it was not responding as expected.
- Working with the Gymnasium, Arcade Learning Environment, and Tensorflow libraries
- Implementing both a neural network and a convolution neural network model
- Implementing an actor-critic model of reinforcement learning.

## About Centipede

### Game summary

Centipede (1981) is a classic, simple shooting game. The player controls a small turret (called an elf in the Atari version) that can move freely along the bottom third of a vertical screen and shoot a bullet upward through a field containing mushrooms (walls) and various insect enemies. The player is allowed only one bullet on screen at a time. If the turret collides with any insect, it ends the life the field resets.

Each insect has their own movement patterns and impacts on the gambe board.

- The namesake centipede is made up of multiple segments that move together in a line, strafing back and forth across the top of the screen and moving downward every time the centipede hits a screen edge, mushroom, or another centipede. If the centipede hits a poisonous mushroom, it becomes enraged and travels straight downward until the head is destroyed.
- Spiders move across the lower third of the screen from one side to another in a random, vertical/diagonal zigzag pattenr
- Scorpions cross a single line of the field horizontally, poisoning any mushrooms that they encounter in the line.
- Fleas drop from the top of the screen to the bottom vertically, leaving mushrooms randomly behind them.

If the player destroys all segments of a centipede, the color scheme of the board changes and a new "level" begins, with new centipede(s) entering from the top of the screen.

Each time a player successfully shoots an insect it is destroyed and the player is awarded varying amounts of points. The player also receives a nominal reward for destroying a mushroom, and for every partially destroyd mushroom still on the field at the end of a life.

Complete rules for the Atari 2600 version of the game can be found at https://atariage.com/manual_html_page.php?SoftwareID=911
History and gameplay summary of the arcade version can be found at https://en.wikipedia.org/wiki/Centipede_(video_game)

### Arcade Learning Environment and my choices.

In the Arcade Learning environment version of Centipede, rewards come exclusively in the form of points as scored in the arcade game (https://ale.farama.org/environments/centipede/). Otherwise, the observation feedback comes in the form of a three-color 210 by 160 screenshot of the current rendered frame of the game.

One important challenge to note about Centipede: Because all points are scored when a bullet collides with an enemy, typically a given reward only comes several frames after the action that led to that reward! In this sense, every reward is post-dated to a far future state. Ultimately I did not have an immediate solution to this problem in any of my models, but it is something worth considering when refining reward gamma in learning.

While each game of centipede starts with 3 lives and additional lives can be earned through strong play, I chose to make my learning episodes one-life long for all models in order to shorten training time and ease complexity. I also made significant use of ALE preprocessing wrappers to reduce the complexity of observations for the purposes of faster training.

## Models and methodology

### First Steps

My initial explorations involved using the <a href="https://www.tensorflow.org/tutorials/reinforcement_learning/actor_critic">tensorflow actor-critic tutorial</a> to get cartpole working on my machine. This required several sessons of work, mostly involving installing appropriate packages on my computer and making sure versioning matched I then played around with the working model, making small tweaks, until I was confident to proceed. Though this was not work directly related to AI development on Centipede, it took about half of my time! 

Major steps here included:
- Getting the Cartpole demo running online in Google Colab
- Getting the same demo running in my native development envrionment (Windows 11 with VSCode)
- Adapting that demo from a notebook to a .py file
- Modifying the notebook to play Lunar Lander (another box 2D game) rather than cartpole

Once this was working, the code became my codebase for developing more complicated models involving neural networks and pooling. Before I went in that direction, though, I wanted to experiment with more basic AI techniques, which necessetated building my own Random and Q-learning agents.

### Models explored

As mentioned above, this was an exploratory project for me. I had never worked in this environment or with these types of tools before, so I wanted to explore what was possible given my knowledge of the class. I knew I wanted to build a q-learning reinforcement agent, even though I understood the memory-based limitations of this approach, because I wanted to see how far it would "get" on its own. Additionally, I wanted to work with Deep Learning and neural networks in some form. These were introduced late in the class timeframe, and I had very limited exposure to them before the final weeks, but I knew I wanted to make an attempt. I chose to work with the following different models:

#### Random Agent

<video src="centipede_agents/random-episode-0.mp4" controls width="300" />

This was a simple baseline agent with no learning capabilities, taking entirely random actions, to provide as a baseline comparison to learning procedures. There were two major purposes to this agent. The first was that I used it as a way to understand the ALE interface and how to connect it and extract rewards. The second is that I wanted a baseline to compare to any actual learning-based agents I built.

##### Results

Once up and running, the agent performed as expected. over 100 trials, the random agent averaged a score of 745 points with a min of 62 and a max of 4404.

#### Q-Learning Agent

<video src="centipede_agents/qagent-episode-0.mp4" controls width="300" >

This took a similar approach to the pac-man Q-learning assigments we studied in class. This agent explored various states of the game with an epsilon-greedy approach. Epsilon started at 1 and a decayed over time to 0.1 as the model preferred exploitation.

To get the model to work I had to flatten the (210,180,3) matrix of the observation into a single vector. This vector became the signature for the "state" that the agent explored. From there the agent made a policy or random choice (epsilon greedy) and recorded the resulting q-value in a dictionary that grew as training progressed.

As designed, this Q-learning agent was limited by the amount of RAM available to process its actions. I used a dynamic programming technique to record and update arrays. In practice, I found that my computer could handle only 100 training sessions before running out of space to explore more states.

To free up space for more training, I implemented <a href="https://gymnasium.farama.org/api/wrappers/misc_wrappers/#gymnasium.wrappers.AtariPreprocessing">Atari environment wrappers</a> from ALE. The wrapper is an important tool because it:
- Downscales the observation to (84,84), which means less data from each frame of the game.
- Removed RGB and replaced with grayscale (color did not add much information for the amount of space it took up)
- Stacked frames into groups of 4, a common RL practice. In many games that render at 30+ frames per second, the difference between two adjacent frames might be minimal or inconsequential. Stacking allows the agent to process a larger amount of motion or state change information each observation, and cuts down on the total amount of data needed to process during training.

After wrapping, the agent was able to complete roughly 600 training sessions before running out of memory. At this point the agent was moving more, but still not firing enough to make meaningful progress. I solved this by adding a reward to every action in the action space that involved firing and penalizing moves that did not include a fire action.


##### Results

Q-Learning doesn't work for long! You can see in the video above that the trained agent makes several moves, and then stops dead. This is the result of the agent encountering states it has not yet seen and having no idea what to do. Effectively, the agent "freezes" because it has no information to exploit.

One possible improvement would be to have the agent switch to random behavior if it does not recognize a given state. This would allow the agent to keep moving and (hopefully) eventually find a state it recognizes. However, this isn't really AI learning, it's just placeholder actions when the AI doesn't know what to do. I did not implement this step in favor of exploring other models.


#### Convolution Neural Network and actor-critic model

<video src="centipede_agents/cnnAgent-episode-0.mp4" controls width="300" >

##### The actor-critic model

Implementing a neural network to attempt deep learning on the problem was the "big lift" of this project for me. 

There are several huge differences to this approach and basic Q-learning. The first is that the neural network is performing a machine learning calculation along with gradient descent to optimize a complicated problem; we were working with a much more complicated, multilayered network implementation. The second is that involving neural networks meant I needed to implement a machine learning library, something that I had never done before. Third, I decided to work with a convolution neural network, which requires a 2D input. Effectively, the neural network agent is trying to "see" the screen and interpret it with regression analysis. Convolution smoothes adjacent pixels in the visual images, reducing noise and making it easier for the neural network to detect general patterns in the image.

Perhaps most importantly, I had to move the neural network away from classification (that is, identifying an image in a single category) and toward q-learning. For that purpose I chose an **actor-critic** model.

The basic idea of an actor-critic model is that it uses a neural network with a gradient policy to simultaneously generate two different outputs:
- The "Actor" vector, which represents a policy for selection actions.
- The "Critic" vector (usually a single entry) which estimates the value of the current state.

In practice, the Actor looks toward long-term rewards, and the critic compares it to the immediate consequences to provide feedback. This is effectively a form of temporal difference learning, where the "difference" is the between the Actor's policy choice and the Critic's evaluation of that choice.

More information on the actor-critic models:
https://www.geeksforgeeks.org/machine-learning/actor-critic-algorithm-in-reinforcement-learning/
https://en.wikipedia.org/wiki/Actor-critic_algorithm

##### Implementation

I got it working, but this took a lot of time. Understanding the method of writing the network (and the process of "calling" the network by progressively pushing the results through each layer of the function) was more complicated than I realized. The amount of time it took to get to a working model directly impacted my ability to refine loss and gradient functions, or understand the "gradient tape" methodology of Tensorflow in great detail. 

Most of my modifications centered around sizing the network appropriately for relatively fast learning and appropriate convolution, pooling, and activation functions. 

One major problem with my initial model was that the actor could occasionally recommend actions that were out-of-bounds (that is, suggesting an action that did not exist in the available pool of 18 actions). This was a significant bug that had to be troubleshot to allow for longer training periods.

After understanding the source of the error (out-of-bounds action options) I solved this problem by reviewing the activation functions on the dense layers. I went with Softmax, which creates an even probability distribution across the Actor layer. I also had to use a number of tf.expand_dims() calls to appropriately dimension the observation for a convolution network.

##### Results

Once the model was working, I was able to train the network for 1,000 episodes. It quickly developed an effective policy of moving to the right and shooting from the side, which yielded 2,033 points in the video, an average of roughlty 1,400 points, and scores of 8,000 - 10,000 at its peak. Training was slow at roughly 10 seconds per episode, and so I didn't make a lot of extra attempts to train or modify this iteration of the model. Instead, I worked on optimization to make training faster.

#### Further Refinement - Pooling and penalties


<video src="centipede_agents/cnnAgent_pool5000-episode-0.mp4" controls width="300" >5000 Episode Training


<video src="centipede_agents/cnnAgent_pool10000-episode-10gamma.mp4" controls width="300" >10000 Episode Training

##### Max Pooling

My first refinement to the model was to introduce Max pooling layers into the model. <a href="https://en.wikipedia.org/wiki/Pooling_layer">Max Pooling</a> is a technique in CNNs to aggregate values among adjacent vectors. This has the effect of both further reducing the amount of calcuation needed in the network and spreading the information out throughout the network, increasing the overall connectivity of the frame. The hope is that this makes the action preceptrons more sensitive to the larger attributes of the frame.

##### Results

I chose to add two max pooling layers into the network, creating a (2,2) kernel for pooling. This was the smallest effective pooling I could make. The result was to spead up episode training by a factor of almost 10x, making longer training sessions a much more attainable goal.

I was able to to run a 5000-episode training session as a result of this update. That said, the results were roughly the same. The agent no longer moved, but prioritizing standing still and firing as a strategy.

##### Penalties and gamma alteration

I was frustrated that the agent seemed to be picking a "plant in one place and fire" philosophy. While effective, and achieving rewards, it wasn't playing like a player so much as a short heuristic. A better policy would be "move to the corner and shoot, but also move out of the way of any enemies coming directly at you" for example, and skilled human players could easily outperform this model.. There are two likely causes to this problem:

- It's likely that part of the problem here is the objective function and the limited reward data available to the agent (points are the only thing that matters). The objective function is probably overvaluing scoring points against survival, so the agent does not see survival as a necessary maneuver.
- It's possible that the loss function is not optimized for Centipede. While this solution probably represents a local minimum, it's unlikely to be the global minimum.

At this point I was running out of time to finish the project at this point, and I didn't have a lot of time to examine the objective function. Instead I opted for a couple attempts at "quick fixes" to see if they would make a difference.

First, I implemented a penalty system that would penalize the agent for taking the same action repeatedly. Honestly I don't think I ever got the code working for this correctly though. The penalized reward did not seem feed back into the model (or the long-term rewards for repeated movement were too great), and the policy converged to the same actions (although sometimes it would move to the upper corner of one side or the other and shoot. I was also worried about repeated cycles of movement (such as the repeated tuple (no-action,shoot)), so I created a version that penalized the agent if it took any of the last 3 actions at the same time. This did not appear to have any effect on performance either.

My next attempt at discouraging the same pattern of movement was to decrease the gamma, or value of future rewards. My hope was that devaluing the long-term reward of shooting at a distant entity would not be as effective as immediate survival. However, with 10,000 episodes of training on a gamma of .1, the agent still behaved the same, running to the corner and shooting. Try as I might, I was not able to change this core behavior in the time I had left.

## Conclusion

Centipede is an interesting game for a RL project in the sense that it has simple, understandable rules, and a reasonable "entry level" strategy. All of my agents were able to find the entry level strategy, but none of them were able to move beyond it in the time alloted. I suspect that the core barrier to my agent after the many attempts are that I still haven't built the right tools for the job. The good news is that this leaves a lot of opportunity for future improvmeents.

Regardless of the success of the agents themselves, I have learned a lot in this project, including how to set up and utilize external libraries, and some early basic work with machine learning. I've enjoyed AI as a topic and it's clear that there is a lot more for me to explore.

### Future Improvements

Future improvements to this project would include:

- Altering the Objective function to value staying alive at least as much as scoring points. This could involve developing a heuristic for distance from enemies, or simply a "living bonus" for future states. This is the area most likely to bring about a change in agent behavior.
- Close examination of the loss function to verify that it is providing the right data to the agent and appropriately following gradient descent to an absolute maximum.
- Further developing a DQN or Deep Q-Learning network.
- Further refining the size of the convolution network layers to be more appropriate for the size and complexity of data.
- Longer training sessions after all of these features are implemented.

# Acknowledgements

Huge thanks to the tensorflow tutorial on actor-critic neural networks for getting me started:  https://www.tensorflow.org/tutorials/reinforcement_learning/actor_critic. 

Additionally, huge thanks to my classmate Thomas Dunn for both moral support throughout the project and a bunch of troubleshooting assistance as I tried to get the appropriate packages loaded onto my computer! This project would not be complete without his support.
